# 58 — JSON Structured Output
**Goal:** Get structured JSON from LLMs using response_format and Pydantic parsing.

Everything downstream — scoring, gap analysis, ranking — wants a dict, not prose. This chapter makes the LLM produce JSON at generation time (`response_format`) and validates it at the boundary with **Pydantic** models, so a malformed or hallucinated response fails loudly instead of corrupting the pipeline.

**Why it matters for resumes / ATS:** free-form LLM text is the classic integration hazard: one run returns `{"skills": [...]}`, the next returns "Here are the skills: ..." in markdown. A structured contract means the extraction stage either yields typed, validated `ResumeSkill` objects or a caught `ValidationError` — never a silent parse failure mid-pipeline.

## 1. Why Structured Output?

Unconstrained LLM output has three recurring failure modes: **format drift** (the shape of the answer changes between calls), **parse failures** (markdown fences, truncated JSON, trailing prose), and **hallucinated field names** (`skill` vs `skills` vs `Skill`). All three are fixable at the generation or validation layer.

**What the code does:** prints the failure modes and the four mitigation strategies, in increasing strictness: prompt-only JSON ("Respond in JSON: ..."), `response_format` (`json_object` / `json_schema` on OpenAI-compatible APIs), Pydantic-based libraries like `instructor`, and constrained generation like `outlines` (which restricts the token sampler itself). The order matters: prompt-only is the cheapest and weakest; constrained generation is the strongest and most work. This chapter uses `response_format` + Pydantic — the pragmatic middle.

In [ ]:
print('''Problems with freeform LLM output:
- Inconsistent formats between calls
- Parsing errors from markdown/incomplete JSON
- Hallucinated field names

Solutions:
1. OpenAI response_format = json_object / json_schema
2. instructor library (Pydantic-based)
3. Outlines (constrained generation)
4. Prompt-only: "Respond in JSON: {skill: ..., years: ...}"''')

## 2. Using response_format

The idea: tell the API *at request time* that the answer must be valid JSON, then parse it into a typed object. `SkillExtract` declares the contract — `skill_name`, `years_experience`, `proficiency` — and doubles as both the schema and the parse target.

**What the code does:**
- Defines `SkillExtract(BaseModel)` — three fields: `str`, `int`, `str`. Pydantic generates the JSON Schema for the model (printed via `model_json_schema(indent=2)`), so the schema lives in one place instead of being hand-copied into the prompt.
- The live call is left commented out because it needs an API key: `client.chat.completions.create(..., response_format={"type": "json_object"})`, then `json.loads(response.choices[0].message.content)`.

**Expected (with a key):** the request returns a JSON object with exactly `skill_name`, `years_experience`, and `proficiency` — for "5 years Python", something like `{"skill_name": "Python", "years_experience": 5, "proficiency": "..."}`. The contract: the model is *encouraged* to emit valid JSON, but nothing guarantees field correctness — that is Pydantic's job in section 3. **Note (verified by running):** on pydantic 2.13 the schema print itself crashes with `TypeError: BaseModel.model_json_schema() got an unexpected keyword argument 'indent'` — drop the `indent=2` argument on that line to see the schema.

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import json

class SkillExtract(BaseModel):
    skill_name: str
    years_experience: int
    proficiency: str

# With a working API key, this would be:
# client = OpenAI(api_key="...", base_url="https://openrouter.ai/api/v1")
# response = client.chat.completions.create(
#     model="openai/gpt-4o-mini",
#     messages=[{"role": "user", "content": "Extract skill from: 5 years Python"}],
#     response_format={"type": "json_object"},
# )
# result = json.loads(response.choices[0].message.content)
# print(result)

print("Structured output schema defined:")
print(SkillExtract.model_json_schema(indent=2))
print("\nWith API key: send request with response_format=json_object")

## 3. Pydantic Parsing

Generation-time hints reduce JSON syntax errors; **Pydantic catches the rest**. `ResumeExtraction` nests a list of `ResumeSkill` objects plus `total_years`, and `model_validate_json()` parses *and* validates in one call — wrong types, missing required fields, and wrong shapes all raise `ValidationError` with a machine-readable list of errors.

**What the code does:**
- `ResumeSkill` — `name: str`, `category: str`, `years: Optional[int] = None`; optional fields tolerate LLMs that omit them.
- `ResumeExtraction` — `skills: List[ResumeSkill]`, `total_years: int`.
- Valid path: `model_validate_json(valid_json)` — **expected (verified by running):** prints `Parsed: Python -> 5 years`.
- Invalid path: `'{"skills": "Python"}'` — `skills` is a string, not a list, so validation fails; **expected (verified by running):** the caught error message is `Input should be a valid array`. The pipeline never sees the bad data — it sees a typed exception it can log and retry.

**Try it:** mutate `valid_json` (drop `total_years`, make `years` a string) and watch each failure surface as a precise Pydantic error instead of a crash downstream.

In [ ]:
from pydantic import BaseModel, ValidationError
from typing import List, Optional

class ResumeSkill(BaseModel):
    name: str
    category: str
    years: Optional[int] = None

class ResumeExtraction(BaseModel):
    skills: List[ResumeSkill]
    total_years: int

# Test parsing
valid_json = '{"skills": [{"name": "Python", "category": "technical"}], "total_years": 5}'
parsed = ResumeExtraction.model_validate_json(valid_json)
print(f"Parsed: {parsed.skills[0].name} -> {parsed.total_years} years")

# Error handling
try:
    bad_json = '{"skills": "Python"}'
    ResumeExtraction.model_validate_json(bad_json)
except ValidationError as e:
    print(f"\nValidation error handled gracefully:")
    print(f"  {e.errors()[0]['msg']}")

## Summary: Structured output is essential for production LLM use. Pydantic parsing catches errors at the boundary.

**Two layers, one contract: ask for JSON at generation time, enforce it at parse time.**

`response_format` reduces syntax errors; Pydantic models turn the response into typed, validated objects and convert the rest into caught `ValidationError`s with exact messages — bad shapes become events you can log and retry, never silent corruption. Because the schema is generated from the model, the contract cannot drift between the prompt and the parser.

Structured output is the plumbing Ch. 59 builds on: tool calling hands the LLM typed arguments, and the same Pydantic discipline applies to tool-call payloads.